In [1]:
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# logging.basicConfig(level=logging.DEBUG)
n_epoch = 1000
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.Newton(model=model, lr_init=1e-2, solver="cg-trunc")


model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.6525646448135376
epoch:  1, loss: 0.6402007937431335
epoch:  2, loss: 0.628068745136261
epoch:  3, loss: 0.6161645650863647
epoch:  4, loss: 0.6044848561286926
epoch:  5, loss: 0.5930259227752686
epoch:  6, loss: 0.5817843079566956
epoch:  7, loss: 0.5707563161849976
epoch:  8, loss: 0.5599386096000671
epoch:  9, loss: 0.549327552318573
epoch:  10, loss: 0.5389198064804077
epoch:  11, loss: 0.5287120938301086
epoch:  12, loss: 0.5187007784843445
epoch:  13, loss: 0.5088827013969421
epoch:  14, loss: 0.4992545247077942
epoch:  15, loss: 0.489812970161438
epoch:  16, loss: 0.4805550277233124
epoch:  17, loss: 0.4714770019054413
epoch:  18, loss: 0.46257632970809937
epoch:  19, loss: 0.4538498818874359
epoch:  20, loss: 0.4452941417694092
epoch:  21, loss: 0.43690651655197144
epoch:  22, loss: 0.42868372797966003
epoch:  23, loss: 0.42062294483184814
epoch:  24, loss: 0.4127214550971985
epoch:  25, loss: 0.4049794375896454
epoch:  26, loss: 0.39738914370536804
epoch:  2

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -362.4662419984886
Test metrics:  R2 = -369.5750948966558


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, solver="cg-trunc", line_search_cond="armijo", line_search_method="backtrack", tau=0.1)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.5995899438858032
epoch:  1, loss: 0.1455405205488205
epoch:  2, loss: 0.03233858197927475
epoch:  3, loss: 0.026555990800261497
epoch:  4, loss: 0.01670972630381584
epoch:  5, loss: 0.013973920606076717
epoch:  6, loss: 0.009986485354602337
epoch:  7, loss: 0.009127747267484665
epoch:  8, loss: 0.008692817762494087
epoch:  9, loss: 0.00843974482268095
epoch:  10, loss: 0.008383460342884064
epoch:  11, loss: 0.008132997900247574
epoch:  12, loss: 0.007925289683043957
epoch:  13, loss: 0.007830778136849403
epoch:  14, loss: 0.007753838784992695
epoch:  15, loss: 0.0076993596740067005
epoch:  16, loss: 0.007523140404373407
epoch:  17, loss: 0.007271832320839167
epoch:  18, loss: 0.006963114719837904
epoch:  19, loss: 0.006692219525575638
epoch:  20, loss: 0.006418138742446899
epoch:  21, loss: 0.006032456178218126
epoch:  22, loss: 0.005115674342960119
epoch:  23, loss: 0.004421397112309933
epoch:  24, loss: 0.003118257038295269
epoch:  25, loss: 0.0025182929821312428
e

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9975324262422266
Test metrics:  R2 = 0.9958550196439502


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonTR(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=n_epoch,
    max_patience=50
)

epoch:  0, loss: 0.5450339317321777
epoch:  1, loss: 0.03317955881357193
epoch:  2, loss: 0.03292504698038101
epoch:  3, loss: 0.032724473625421524
epoch:  4, loss: 0.03253637254238129
epoch:  5, loss: 0.0323907807469368
epoch:  6, loss: 0.03224542737007141
epoch:  7, loss: 0.032139938324689865
epoch:  8, loss: 0.0320051871240139
epoch:  9, loss: 0.03190649300813675
epoch:  10, loss: 0.03176717460155487
epoch:  11, loss: 0.03165125101804733
epoch:  12, loss: 0.03150321915745735
epoch:  13, loss: 0.03136583790183067
epoch:  14, loss: 0.0312185175716877
epoch:  15, loss: 0.031074263155460358
epoch:  16, loss: 0.03091920167207718
epoch:  17, loss: 0.0307676550000906
epoch:  18, loss: 0.030604880303144455
epoch:  19, loss: 0.030442342162132263
epoch:  20, loss: 0.030267709866166115
epoch:  21, loss: 0.03009723126888275
epoch:  22, loss: 0.029914364218711853
epoch:  23, loss: 0.02973564714193344
epoch:  24, loss: 0.02954407036304474
epoch:  25, loss: 0.029355617240071297
epoch:  26, loss: 0

In [10]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.6734427069270424
Test metrics:  R2 = 0.6590024936251906
